In [1]:
pip install transformers datasets evaluate -qq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00


# Question Answering Model Fine tuning

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import DefaultDataCollator
from transformers import AutoModelForQuestionAnswering, TrainingArguments, Trainer
from transformers import pipeline

In [3]:
squad = load_dataset("rajpurkar/squad", split="train[:5000]")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 14.5MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 1.82MB            

plain_text/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [4]:
squad = squad.train_test_split(test_size=0.1)

In [5]:
squad["train"][0]

{'id': '56cbdcd16d243a140015eda4',
 'title': 'Frédéric_Chopin',
 'context': "In October 1810, six months after Fryderyk's birth, the family moved to Warsaw, where his father acquired a post teaching French at the Warsaw Lyceum, then housed in the Saxon Palace. Fryderyk lived with his family in the Palace grounds. The father played the flute and violin; the mother played the piano and gave lessons to boys in the boarding house that the Chopins kept. Chopin was of slight build, and even in early childhood was prone to illnesses.",
 'question': 'Where did Frédéric live with his family while they were in Warsaw?',
 'answers': {'text': ['the Palace grounds'], 'answer_start': [218]}}

In [6]:
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [7]:
def preprocess_function(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=384,
        truncation="only_second",
        return_offsets_mapping=True,
        padding="max_length",
    )
 
    offset_mapping = inputs.pop("offset_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []
 
    for i, offset in enumerate(offset_mapping):
        answer = answers[i]
        start_char = answer["answer_start"][0]
        end_char = answer["answer_start"][0] + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)
 
        # Find the start and end of the context
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1
 
        # If the answer is not fully inside the context, label it (0, 0)
        if offset[context_start][0] > end_char or offset[context_end][1] < start_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # Otherwise it's the start and end token positions
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)
 
            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)
 
    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

In [8]:
tokenized_squad = squad.map(
    preprocess_function,
    batched=True,
    remove_columns=squad["train"].column_names
)

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [9]:
data_collator = DefaultDataCollator()

In [10]:
model = AutoModelForQuestionAnswering.from_pretrained("distilbert/distilbert-base-uncased")

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForQuestionAnswering LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
qa_outputs.bias         | MISSING    | 
qa_outputs.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    report_to="none"
)

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_squad["train"],
    eval_dataset=tokenized_squad["test"],
    processing_class=tokenizer,
    data_collator=data_collator
)

In [13]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,2.196320
2,2.782036,1.623468
3,2.782036,1.550436


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=846, training_loss=2.212942028722019, metrics={'train_runtime': 551.1689, 'train_samples_per_second': 24.493, 'train_steps_per_second': 1.535, 'total_flos': 1322862637824000.0, 'train_loss': 2.212942028722019, 'epoch': 3.0})

In [14]:
trainer.save_model("qa_model")
tokenizer.save_pretrained("qa_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('qa_model/tokenizer_config.json', 'qa_model/tokenizer.json')

In [19]:
qa_model = AutoModelForQuestionAnswering.from_pretrained("qa_model")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [20]:
tok = AutoTokenizer.from_pretrained("qa_model")

In [22]:
question = "what are the steps to make a jet engine?"

context = """
Building a scale model of a jet engine requires a clear understanding of its core components and assembly order.Here is a structural breakdown of the major sections and assembly steps to help you build or describe your model.Core Sections of a Jet EngineA standard turbojet engine consists of five primary sections, arranged from front to back:Inlet (Intake): Funnels incoming air into the compressor.Compressor: Rows of spinning blades that squeeze and pressurize the air.Combustion Chamber (Combustor): Mixing zone where fuel injects into the pressurized air and ignites.Turbine: Rows of blades that catch the energy of escaping hot gases to spin the compressor shaft.Nozzle (Exhaust): Narrows the exiting hot air to create high-velocity thrust.Step-by-Step Model Assembly Order1. The Central Shaft (The Core)Function: Connects the turbine at the back to the compressor at the front.Model Step: Start with a central rod or axle. This acts as the structural spine for all rotating parts.
"""

In [16]:
question_answerer = pipeline("question-answering", model="qa_model", tokenizer="qa_model")

KeyError: "Unknown task question-answering, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

In [17]:
question_answerer(question=question, context=context)

NameError: name 'question_answerer' is not defined

In [26]:
def answer_question(question, context, model, tokenizer, max_answer_length=30):
    inputs = tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation=True,
    )
 
    with torch.no_grad():
        outputs = model(**inputs)
 
    start_logits = outputs.start_logits[0]
    end_logits = outputs.end_logits[0]
 
    best_score = float("-inf")
    best_start = 0
    best_end = 0
 
    for start in range(len(start_logits)):
        for end in range(start, min(start + max_answer_length, len(end_logits))):
            score = start_logits[start].item() + end_logits[end].item()
 
            if score > best_score:
                best_score = score
                best_start = start
                best_end = end
 
    answer_tokens = inputs["input_ids"][0][best_start:best_end + 1]
 
    return tokenizer.decode(
        answer_tokens,
        skip_special_tokens=True
    ), best_score
 
import torch
print(answer_question(question, context, qa_model, tok))

('july 20, 1969', 11.670998573303223)


In [27]:
# evaluation_dataset.py

qa_evaluation_dataset = [
    {
        "id": "item_001",
        "category": "Extractive / Fact-based",
        "context": "The Apollo 11 mission landed the first humans on the Moon on July 20, 1969. Commander Neil Armstrong and Lunar Module Pilot Buzz Aldrin formed the American crew that detoured into history.",
        "question": "When did the Apollo 11 mission land on the Moon?",
        "expected_answers": ["July 20, 1969", "1969", "July 20"]
    },
    {
        "id": "item_002",
        "category": "Numerical Reasoning",
        "context": "The company reported a revenue of $4.2 billion in Q1. In Q2, the revenue increased by $1.1 billion due to high demand, before dropping by $0.5 billion in Q3.",
        "question": "What was the total revenue reported by the company in Q2?",
        "expected_answers": ["$5.3 billion", "5.3 billion", "5.3"]
    },
    {
        "id": "item_003",
        "category": "Inference / Synthesis",
        "context": "Water boils at 100 degrees Celsius at sea level. However, higher atmospheric pressures increase the boiling point, while lower pressures decrease it. Quito, Ecuador sits at a high altitude with low atmospheric pressure.",
        "question": "Would water boil at a higher or lower temperature in Quito compared to sea level?",
        "expected_answers": ["lower temperature", "lower", "at a lower temperature"]
    },
    {
        "id": "item_004",
        "category": "Unanswerable / Adversarial",
        "context": "Photosynthesis is a process used by plants and other organisms to convert light energy into chemical energy. This chemical energy is stored in carbohydrate molecules like sugars.",
        "question": "Who discovered the exact molecular structure of chlorophyll?",
        "expected_answers": ["", "unanswerable", "Not mentioned in context"] # Evaluates if model hallucinates
    }
]

In [28]:
for example in qa_evaluation_dataset:
    context = example["context"]
    question = example["question"]
    expected_answers = example["expected_answers"]
    
    predicted_answer, score = answer_question(question, context, qa_model, tok)
    
    print(f"Question: {question}")
    print(f"Predicted Answer: {predicted_answer}")
    print(f"Expected Answers: {expected_answers}")
    print(f"Match: {predicted_answer in expected_answers}")
    print(f"Confidence Score: {score}")
    print("-" * 50)

Question: When did the Apollo 11 mission land on the Moon?
Predicted Answer: july 20, 1969
Expected Answers: ['July 20, 1969', '1969', 'July 20']
Match: False
Confidence Score: 11.670998573303223
--------------------------------------------------
Question: What was the total revenue reported by the company in Q2?
Predicted Answer: $ 4. 2 billion
Expected Answers: ['$5.3 billion', '5.3 billion', '5.3']
Match: False
Confidence Score: 11.92585802078247
--------------------------------------------------
Question: Would water boil at a higher or lower temperature in Quito compared to sea level?
Predicted Answer: 100 degrees celsius
Expected Answers: ['lower temperature', 'lower', 'at a lower temperature']
Match: False
Confidence Score: 9.726886749267578
--------------------------------------------------
Question: Who discovered the exact molecular structure of chlorophyll?
Predicted Answer: photosynthesis
Expected Answers: ['', 'unanswerable', 'Not mentioned in context']
Match: False
Confid